# Kafka and Spark Streaming Exercise — Tesla Stock & News

This notebook uses the official exercise structure and completes the Kafka and Spark Streaming requirements for the Tesla financial news and stock project.

Goal: split the merged Tesla stock/news dataset into JSON messages, send the messages to a Kafka topic, and process the topic with Spark Structured Streaming.

**Important for submission:** run all cells on the BDENG cluster/JupyterHub, check that the output cells are visible, and then export the notebook **with output** as PDF.

Use Python, `pandas`, `kafka-python`, `pyspark` and Spark Structured Streaming to send messages to a Kafka topic and analyze them with Spark Streaming.

# Kafka

## Import necessary libraries

The imports are kept small. The optional install block only runs if `kafka-python` is missing in the notebook environment.

In [1]:
import json
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd

try:
    from kafka import KafkaProducer, KafkaConsumer
    from kafka.admin import KafkaAdminClient, NewTopic
    from kafka.errors import TopicAlreadyExistsError
except ModuleNotFoundError:
    import sys
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "kafka-python", "-q"])
    from kafka import KafkaProducer, KafkaConsumer
    from kafka.admin import KafkaAdminClient, NewTopic
    from kafka.errors import TopicAlreadyExistsError

## Load a dataset to stream

The merged result dataset is used because it combines the file-based stock data, scraped Google News data and Alpha Vantage REST API news data. Each row is converted into one JSON message.

In [2]:
DATA_PATH = Path("../data/results/tesla_stock_news_merged.csv")

if not DATA_PATH.exists():
    raise FileNotFoundError(f"Dataset not found: {DATA_PATH.resolve()}")

source_df = pd.read_csv(DATA_PATH)
source_df["Date"] = pd.to_datetime(source_df["Date"], errors="coerce").dt.strftime("%Y-%m-%d")

# Keep all available rows. The dataset is small, so this creates a controlled stream without excessive output.
message_df = source_df.copy()
message_df = message_df.astype(object).where(pd.notna(message_df), None)
messages = message_df.to_dict(orient="records")

print(f"Loaded {len(messages)} rows. Each row will be sent as one Kafka message.")
pd.DataFrame(messages).head(3)

Loaded 18 rows. Each row will be sent as one Kafka message.


,Date,Open,High,Low,Close,Volume,Daily_Return,Price_Range,MA_30,MA_100,scraped_news_count,scraped_sources,scraped_titles,api_news_count,avg_api_sentiment,api_sentiment_labels,api_titles
0,2026-04-06,362.59,367.72,346.64,352.82,77697643,-0.021548,21.08,389.155543,421.931363,1.0,Yahoo Finance,Why JPMorgan is warning Tesla stock may crash ...,0.0,0.0,None,None
1,2026-04-07,346.44,348.02,337.24,346.65,74515355,-0.017488,10.78,387.382877,420.945563,0.0,None,None,0.0,0.0,None,None
2,2026-04-08,363.79,364.50,339.67,343.25,78838616,-0.009808,24.83,385.178543,419.981863,0.0,None,None,0.0,0.0,None,None


## Create a producer and stream the messages

The Kafka topic is created if it does not already exist. Only one summary line is printed to avoid excessive loop output.

In [3]:
# Cluster broker from Moodle. If you run the experimental local Docker setup, use: localhost:9092
KAFKA_BROKER = "172.29.16.101:9092"
TOPIC = "group6_tesla_stock_news_stream_" + datetime.now(timezone.utc).strftime("%Y%m%d%H%M%S")

# Create topic if possible. If the topic already exists, continue.
try:
    admin_client = KafkaAdminClient(bootstrap_servers=KAFKA_BROKER, client_id="tesla_topic_admin")
    topic = NewTopic(name=TOPIC, num_partitions=1, replication_factor=1)
    try:
        admin_client.create_topics(new_topics=[topic], validate_only=False)
        print(f"Created Kafka topic: {TOPIC}")
    except TopicAlreadyExistsError:
        print(f"Kafka topic already exists: {TOPIC}")
    finally:
        admin_client.close()
except Exception as topic_error:
    print(f"Topic creation skipped or failed: {topic_error}")

producer = KafkaProducer(
    bootstrap_servers=KAFKA_BROKER,
    key_serializer=lambda key: str(key).encode("utf-8"),
    value_serializer=lambda value: json.dumps(value, default=str).encode("utf-8"),
)

for message in messages:
    producer.send(TOPIC, key=message.get("Date"), value=message)

producer.flush()
producer.close()

print(f"Sent {len(messages)} messages to Kafka topic '{TOPIC}'.")

Created Kafka topic: group6_tesla_stock_news_stream_20260624143935
Sent 18 messages to Kafka topic 'group6_tesla_stock_news_stream_20260624143935'.


## Create a consumer and check if the messages can be read

This is only a small sanity check. It reads and displays at most five messages, so the notebook output stays short.

In [4]:
consumer = KafkaConsumer(
    TOPIC,
    bootstrap_servers=KAFKA_BROKER,
    auto_offset_reset="earliest",
    enable_auto_commit=False,
    consumer_timeout_ms=5000,
    key_deserializer=lambda key: key.decode("utf-8") if key else None,
    value_deserializer=lambda value: json.loads(value.decode("utf-8")),
)

sample_messages = []
for kafka_message in consumer:
    sample_messages.append({
        "key": kafka_message.key,
        "Date": kafka_message.value.get("Date"),
        "Close": kafka_message.value.get("Close"),
        "Daily_Return": kafka_message.value.get("Daily_Return"),
        "scraped_news_count": kafka_message.value.get("scraped_news_count"),
        "api_news_count": kafka_message.value.get("api_news_count"),
    })
    if len(sample_messages) >= 5:
        break

consumer.close()

print(f"Read {len(sample_messages)} sample messages from Kafka.")
pd.DataFrame(sample_messages)

C:\Users\noahf\AppData\Local\Temp\ipykernel_26844\1040727177.py:1: DeprecationWarning: key_deserializer does not implement kafka.serializer.Deserializer
  consumer = KafkaConsumer(
C:\Users\noahf\AppData\Local\Temp\ipykernel_26844\1040727177.py:1: DeprecationWarning: value_deserializer does not implement kafka.serializer.Deserializer
  consumer = KafkaConsumer(


Read 5 sample messages from Kafka.


,key,Date,Close,Daily_Return,scraped_news_count,api_news_count
0,2026-04-06,2026-04-06,352.82,-0.021548,1.0,0.0
1,2026-04-07,2026-04-07,346.65,-0.017488,0.0,0.0
2,2026-04-08,2026-04-08,343.25,-0.009808,0.0,0.0
3,2026-04-09,2026-04-09,345.62,0.006905,0.0,0.0
4,2026-04-10,2026-04-10,348.95,0.009635,0.0,0.0


# Kafka and Spark Streaming

Spark acts as a Kafka consumer. The stream is parsed from JSON, transformed and aggregated into weekly summaries.

## Spark Context and Session

The log level is set to `ERROR` to prevent unnecessary Spark logs in the exported PDF.

In [5]:
import os

os.environ["JAVA_HOME"] = r"C:\Program Files\Eclipse Adoptium\jdk-17.0.15.6-hotspot"
os.environ["PATH"] = os.environ["JAVA_HOME"] + r"\bin;" + os.environ["PATH"]

In [6]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("TeslaKafkaSparkStreaming")
    .master("local[*]")
    .config(
        "spark.jars.packages",
        "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.1"
    )
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

KeyboardInterrupt: 

## Create a Spark DataFrame from a Kafka stream

The Kafka stream starts at the earliest available offset, so the messages sent above are included in the streaming query.

In [ ]:
kafka_stream_df = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", KAFKA_BROKER)
    .option("subscribe", TOPIC)
    .option("startingOffsets", "earliest")
    .option("failOnDataLoss", "false")
    .load()
)

kafka_stream_df.printSchema()

## Convert the binary Kafka data to strings

Kafka stores keys and values as binary data. For JSON parsing, the value column is converted to a string.

In [ ]:
json_stream_df = kafka_stream_df.selectExpr(
    "CAST(key AS STRING) AS kafka_key",
    "CAST(value AS STRING) AS json_value",
    "timestamp AS kafka_timestamp"
)

json_stream_df.printSchema()

## Create a structured schema for the streamed data

The schema matches the columns of `tesla_stock_news_merged.csv`. After parsing, additional columns for event time, total news count and return direction are created.

In [ ]:
tesla_schema = StructType([
    StructField("Date", StringType(), True),
    StructField("Open", DoubleType(), True),
    StructField("High", DoubleType(), True),
    StructField("Low", DoubleType(), True),
    StructField("Close", DoubleType(), True),
    StructField("Volume", LongType(), True),
    StructField("Daily_Return", DoubleType(), True),
    StructField("Price_Range", DoubleType(), True),
    StructField("MA_30", DoubleType(), True),
    StructField("MA_100", DoubleType(), True),
    StructField("scraped_news_count", DoubleType(), True),
    StructField("scraped_sources", StringType(), True),
    StructField("scraped_titles", StringType(), True),
    StructField("api_news_count", DoubleType(), True),
    StructField("avg_api_sentiment", DoubleType(), True),
    StructField("api_sentiment_labels", StringType(), True),
    StructField("api_titles", StringType(), True),
])

parsed_stream_df = (
    json_stream_df
    .select(
        col("kafka_key"),
        from_json(col("json_value"), tesla_schema).alias("data"),
        col("kafka_timestamp"),
    )
    .select("kafka_key", "data.*", "kafka_timestamp")
    .withColumn("event_time", to_timestamp(col("Date"), "yyyy-MM-dd"))
    .withColumn("total_news_count", col("scraped_news_count") + col("api_news_count"))
    .withColumn(
        "return_direction",
        when(col("Daily_Return") > 0, lit("positive"))
        .when(col("Daily_Return") < 0, lit("negative"))
        .otherwise(lit("neutral"))
    )
)

parsed_stream_df.printSchema()

## Create a DataFrame grouped by a time window

The stream is aggregated into seven-day windows. For each window and return direction, the number of messages, average return, average close price, average volume and average news count are calculated.

In [ ]:
windowed_stream_df = (
    parsed_stream_df
    .withWatermark("event_time", "14 days")
    .groupBy(window(col("event_time"), "7 days"), col("return_direction"))
    .agg(
        count("*").alias("message_count"),
        avg("Daily_Return").alias("avg_daily_return"),
        avg("Close").alias("avg_close"),
        avg("Volume").alias("avg_volume"),
        avg("total_news_count").alias("avg_total_news_count"),
    )
)

## Create a query stream of the DataFrame

The output is written to a memory table. `trigger(once=True)` processes the available Kafka messages once and then stops, which is suitable for a controlled exercise notebook.

In [ ]:
query = (
    windowed_stream_df.writeStream
    .format("memory")
    .queryName("tesla_stream_summary")
    .outputMode("complete")
    .trigger(once=True)
    .start()
)

query.awaitTermination(timeout=60)

summary_spark_df = spark.sql("""
    SELECT
        window.start AS window_start,
        window.end AS window_end,
        return_direction,
        message_count,
        ROUND(avg_daily_return, 5) AS avg_daily_return,
        ROUND(avg_close, 2) AS avg_close,
        ROUND(avg_volume, 0) AS avg_volume,
        ROUND(avg_total_news_count, 2) AS avg_total_news_count
    FROM tesla_stream_summary
    ORDER BY window_start, return_direction
""")

summary_spark_df.show(20, truncate=False)

## Export the processed data as a Pandas DataFrame and visualize it

The aggregated streaming result is saved as a CSV file for later presentation and visualized with a compact chart.

In [ ]:
import matplotlib.pyplot as plt

summary_pd = summary_spark_df.toPandas()

output_path = Path("../data/results/tesla_kafka_stream_summary.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)
summary_pd.to_csv(output_path, index=False)

print(f"Saved {len(summary_pd)} aggregated rows to {output_path}")
summary_pd.head(10)

## Visualization: streaming result

This chart shows how many streamed messages occurred per seven-day window and return direction.

In [ ]:
if summary_pd.empty:
    print("No streaming summary available. Check Kafka topic, broker and Spark Kafka connector.")
else:
    plot_df = summary_pd.copy()
    plot_df["window_start"] = pd.to_datetime(plot_df["window_start"])
    pivot_df = plot_df.pivot_table(
        index="window_start",
        columns="return_direction",
        values="message_count",
        aggfunc="sum",
        fill_value=0,
    )

    ax = pivot_df.plot(kind="bar", figsize=(10, 5))
    ax.set_title("Tesla Kafka Stream: Messages per 7-day Window")
    ax.set_xlabel("Window start")
    ax.set_ylabel("Message count")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

## Short conclusion

The merged Tesla dataset was split into individual JSON messages and written to a Kafka topic. Spark Structured Streaming consumed the Kafka topic, parsed the messages with a defined schema and created weekly aggregates. This demonstrates the complete pipeline required for the exercise: dataset → messages → Kafka topic → Spark Streaming processing → result file and visualization.